# Détection automatique de faux scans et d'anomalies dans les bases d'imagerie médicale
### Approche hybride apprentissage profond + forensique numérique — carnet d'expérimentation (LIDC-IDRI)

**Auteur :** Ing. Mendoua Vinny Jordan — Master Recherche SDIA, ENSPD, 2025-2026

Ce carnet implémente le **protocole complet** décrit au chapitre 2 du mémoire :

| § | Contenu |
|---|---|
| 0 | Installation, configuration, graines, GPU |
| 1 | Lecture de LIDC-IDRI (pydicom / pylidc), conversion en HU, masque pulmonaire |
| 2 | Partition 70/15/15 **par patient**, inventaire des coupes |
| 3 | Attaques classiques : copier-déplacer, inpainting, bruit, FGSM (+ camouflage du bruit) |
| 4 | GAN d'inpainting conditionnel (pix2pix) : injection / suppression de nodules |
| 5 | DDPM + inpainting de type RePaint |
| 5 bis | Construction du jeu falsifié (coupes + masques de vérité terrain) |
| 6 | Branche forensique (résidus SRM/KV, autocorrélation, spectre, incohérence locale, duplication) |
| 7 | Branche CNN : couche contrainte de Bayar + SRM → EfficientNet-B0 → U-Net, perte BCE + Dice |
| 8 | Branche diffusion : score d'erreur de reconstruction |
| 9 | Fusion local → global, évaluation (test interne, CT-GAN externe, robustesse) et **génération du tableau 3.6** |
| 10 | Module de sécurité hospitalier : SHA-256, journal d'audit chaîné, politique d'alerte, interface Gradio |
| 11 | Scripts de la preuve de concept SimCT-Tamper (reproduction des chiffres du chapitre 3) |
| 12 | Figures calculées (résidus, bruit, copier-déplacer), sur une coupe réelle LIDC-IDRI si disponible |

**Matériel conseillé :** GPU ≥ 8 Go : Kaggle / Colab (T4, P100) ou GPU local, y compris AMD Radeon RX 7600 sous Windows avec PyTorch ROCm. Sur Kaggle : activer **GPU** et **Internet**, puis « Save & Run All ». Les scanners LIDC-IDRI retenus sont **téléchargés automatiquement depuis le TCIA**, lus puis supprimés au fur et à mesure (l'espace disque reste sous quelques Go). L'ensemble tourne en 3 à 5 heures avec 300 patients.

> Règle forensique fondamentale : **ne jamais rééchantillonner ni exporter en 8 bits** les images avant analyse (cf. §3.1.4 du mémoire). Toutes les opérations travaillent sur les HU natifs 16 bits.

## 0. Installation et configuration

In [ ]:
# Installation automatique des paquets manquants (Kaggle / Colab / local)
import importlib.util, subprocess, sys
_need = {"pydicom": "pydicom", "pylidc": "pylidc", "SimpleITK": "SimpleITK", "timm": "timm", "skimage": "scikit-image",
         "cv2": "opencv-python-headless", "tabulate": "tabulate", "gdown": "gdown"}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("installation :", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=False)

In [ ]:
import os, json, math, time, random, hashlib, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from scipy.stats import kurtosis
import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings("ignore")

# Racine des données : /kaggle/working sur Kaggle, /content sur Colab, Documents\GPU_ROCm sur un PC Windows local
if Path("/kaggle/working").exists():   BASE = Path("/kaggle/working")
elif Path("/content").exists():        BASE = Path("/content")
else:                                  BASE = Path.home() / "Documents" / "GPU_ROCm"
CFG = dict(
    LIDC_ROOT=str(BASE / "LIDC-IDRI"),   # DICOM LIDC-IDRI (téléchargés automatiquement depuis le TCIA si absents)
    CTGAN_ROOT=str(BASE / "CT-GAN"),     # jeu UCI id=520 (test externe), téléchargé si possible
    OUT=str(BASE / "out"),
    FAST=True,                           # sous-ensemble de patients
    MAX_PATIENTS=300,                    # utilisé si FAST (300 patients : 210 entraînement, 45 validation, 45 test)
    DOWNLOAD_LIDC=True,                  # télécharger depuis le TCIA les scanners absents
    KEEP_DICOM=False,                    # supprimer chaque scanner après lecture (économie de disque)
    DOWNLOAD_CTGAN=True,                 # tenter de télécharger le jeu CT-GAN (6 Go) pour le test externe
    N_JOBS=max(1, (os.cpu_count() or 2)),
    SLICES_PER_PATIENT=12,
    P_AUTH=0.5,                          # proportion de coupes authentiques
    PATCH=64, STRIDE=32,
    SEED=42,
    GAN_EPOCHS=30, DDPM_EPOCHS=40, CNN_EPOCHS=30, NODCLS_EPOCHS=3,
    BATCH=128, LR=3e-4,
    T_DIFF=1000, T_STAR=150,
)
OUT = Path(CFG["OUT"]); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "slices").mkdir(exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else 2   # sous Windows, les workers DataLoader ne voient pas les classes du notebook
T_START = time.time()
def chrono(msg): print(f"[{(time.time() - T_START) / 60:6.1f} min] {msg}", flush=True)

def pmap(fn, items, chunksize=4):
    # Parallélisation CPU (fork sous Linux) ; repli séquentiel sous Windows ou en cas d'erreur.
    items = list(items)
    if os.name != "nt" and CFG["N_JOBS"] > 1 and len(items) > 8:
        try:
            import multiprocessing as mp
            with mp.get_context("fork").Pool(CFG["N_JOBS"]) as pool:
                return pool.map_async(fn, items, chunksize=chunksize).get(timeout=4 * 3600)
        except Exception as e:
            print("parallélisation indisponible, passage séquentiel :", e)
    return [fn(it) for it in items]
print("Périphérique :", DEVICE, "| PyTorch", torch.__version__,
      "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")

# Radeon sous Windows (ROCm) : la BatchNorm de MIOpen doit être compilée à la volée et cette
# compilation échoue (« type_traits file not found »). On force la BatchNorm native de PyTorch
# et on garde MIOpen pour les convolutions. Le premier passage est lent (recherche des noyaux
# MIOpen, quelques minutes), les suivants utilisent le cache.
if DEVICE.type == "cuda" and getattr(torch.version, "hip", None) and os.name == "nt":
    _bn_miopen = F.batch_norm
    def _bn_native(*args, **kwargs):
        with torch.backends.cudnn.flags(enabled=False):
            return _bn_miopen(*args, **kwargs)
    F.batch_norm = _bn_native
    print("Correctif ROCm Windows actif : BatchNorm native, convolutions MIOpen.")

## 1. Lecture de LIDC-IDRI et prétraitement

Les volumes sont lus avec **pylidc**, qui gère l'ordre des coupes et la conversion en unités Hounsfield (pente/ordonnée DICOM).
Les nodules retenus sont ceux annotés par **au moins 3 radiologues** (regroupement `cluster_annotations`).

In [ ]:
# Compatibilité pylidc / NumPy récents
for _a, _t in (("int", int), ("float", float), ("bool", bool)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)
rc = Path.home() / (".pylidcrc" if os.name != "nt" else "pylidc.conf")
rc.write_text(f"[dicom]\npath = {CFG['LIDC_ROOT']}\nwarn = True\n")
import pylidc as pl

scans = pl.query(pl.Scan).all()
patients = sorted({s.patient_id for s in scans})
if CFG["FAST"]:
    rng = np.random.default_rng(CFG["SEED"])
    patients = sorted(rng.choice(patients, size=min(CFG["MAX_PATIENTS"], len(patients)), replace=False).tolist())
scan_of = {}
for s in scans:
    if s.patient_id in patients and s.patient_id not in scan_of:
        scan_of[s.patient_id] = s
patients = [p for p in patients if p in scan_of]
print(len(scan_of), "patients retenus")

# --- Téléchargement à la demande depuis le TCIA (API publique NBIA) ---
import io, zipfile, shutil, urllib.request, urllib.parse
NBIA = "https://services.cancerimagingarchive.net/nbia-api"
_TOKEN = {}
def _nbia_token():
    if "v" not in _TOKEN:
        data = urllib.parse.urlencode({"username": "nbia_guest", "password": "", "client_id": "NBIA", "grant_type": "password"}).encode()
        with urllib.request.urlopen(urllib.request.Request(NBIA + "/oauth/token", data=data), timeout=60) as r:
            _TOKEN["v"] = json.load(r)["access_token"]
    return _TOKEN["v"]

def series_dir(pid):
    return Path(CFG["LIDC_ROOT"]) / pid / scan_of[pid].series_instance_uid

def ensure_dicom(pid):
    d = series_dir(pid)
    if (d / ".ok").exists() or not CFG["DOWNLOAD_LIDC"]:
        return d
    uid, last = scan_of[pid].series_instance_uid, None
    for attempt in range(4):
        for url, auth in ((NBIA + "/services/v1/getImage?SeriesInstanceUID=" + uid, False),
                          (NBIA + "/services/v2/getImage?SeriesInstanceUID=" + uid, True)):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "lidc-tampering-research"})
                if auth: req.add_header("Authorization", "Bearer " + _nbia_token())
                with urllib.request.urlopen(req, timeout=300) as r:
                    data = r.read()
                d.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(io.BytesIO(data)) as z:
                    for m in z.infolist():
                        if m.filename.lower().endswith(".dcm"):
                            (d / Path(m.filename).name).write_bytes(z.read(m))
                (d / ".ok").write_text("ok")
                return d
            except Exception as e:
                last = e
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"téléchargement impossible pour {pid} : {last}")

def drop_dicom(pid):
    if not CFG["KEEP_DICOM"]:
        shutil.rmtree(Path(CFG["LIDC_ROOT"]) / pid, ignore_errors=True)

In [ ]:
def lung_mask_slice(hu):
    # Segmentation pulmonaire simple : air interne (< -400 HU) non connecté au bord, 2 plus grandes composantes.
    m = hu < -400
    lab, n = ndi.label(m)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    m = m & ~np.isin(lab, border)
    lab, n = ndi.label(m)
    if n == 0:
        return m
    sizes = ndi.sum(m, lab, range(1, n + 1))
    order = np.argsort(sizes)[::-1][:2]
    keep = [i + 1 for i in order if sizes[i] > 1500]
    m = np.isin(lab, keep)
    m = ndi.binary_closing(m, iterations=3)
    return ndi.binary_fill_holes(m)

def load_patient(pid):
    ensure_dicom(pid)
    s = scan_of[pid]
    vol = s.to_volume(verbose=False).astype(np.int16)          # (512, 512, Z) en HU
    nods = []
    for cl in s.cluster_annotations():
        if len(cl) >= 3:
            c = np.mean([a.centroid for a in cl], axis=0)       # (ligne, colonne, coupe)
            d = float(np.mean([a.diameter for a in cl]))
            nods.append((int(round(c[0])), int(round(c[1])), int(round(c[2])), d / s.pixel_spacing / 2))
    return vol, nods, float(s.pixel_spacing), float(s.slice_thickness)

vol, nods, ps, st = load_patient(patients[0])
z = nods[0][2] if nods else vol.shape[2] // 2
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(np.clip((vol[:, :, z] + 1350) / 1500, 0, 1), cmap="gray"); ax[0].set_title(f"{patients[0]} coupe {z}")
ax[1].imshow(lung_mask_slice(vol[:, :, z]), cmap="gray"); ax[1].set_title("Masque pulmonaire")
for a in ax: a.axis("off")
plt.show(); print("nodules (≥3 lecteurs) :", nods)

## 2. Partition par patient et inventaire des coupes

Aucune coupe d'un patient du test n'est vue à l'entraînement (coupes d'un même patient = même dose, même noyau, même anatomie).

In [ ]:
rng = np.random.default_rng(CFG["SEED"])
perm = rng.permutation(patients)
n = len(perm); n_tr = int(0.70 * n); n_va = int(0.15 * n)
SPLIT = {p: "train" for p in perm[:n_tr]}
SPLIT.update({p: "val" for p in perm[n_tr:n_tr + n_va]})
SPLIT.update({p: "test" for p in perm[n_tr + n_va:]})
print(pd.Series(SPLIT).value_counts())

def candidate_slices(vol, nods, k, rng):
    # Coupes riches en poumon ; on garantit la présence des coupes à nodule (pour les attaques de suppression).
    Z = vol.shape[2]
    area = np.array([(vol[::4, ::4, z] < -400).mean() for z in range(Z)])
    good = np.where(area > 0.12)[0]
    nod_z = sorted({c[2] for c in nods if 0 <= c[2] < Z})
    rest = [z for z in good if z not in nod_z]
    pick = list(nod_z[: k // 2]) + list(rng.choice(rest, size=min(len(rest), k - min(len(nod_z), k // 2)), replace=False))
    return sorted(int(z) for z in pick)

## 3. Attaques classiques et camouflage du bruit

L'attaquant estime le niveau de bruit local (MAD du résidu médian, éq. 2.1) et ajoute un bruit blanc de même écart-type au contenu inséré, fondu par une fenêtre gaussienne (éq. 2.2).

In [ ]:
HU_MIN, HU_MAX = -1024, 3071

def mad_sigma(img, mask):
    f = img.astype(np.float32)
    r = f - ndi.median_filter(f, 3)
    v = r[mask] if mask.any() else r.ravel()
    return 1.4826 * np.median(np.abs(v - np.median(v))) * 1.2

def blend_patch(img, patch, cy, cx, sigma, rng):
    out = img.astype(np.float32).copy()
    h, w = patch.shape
    y0, x0 = cy - h // 2, cx - w // 2
    yy, xx = np.mgrid[:h, :w]
    win = np.exp(-(((yy - h / 2) / (h / 3.2)) ** 2 + ((xx - w / 2) / (w / 3.2)) ** 2))
    reg = out[y0:y0 + h, x0:x0 + w]
    out[y0:y0 + h, x0:x0 + w] = reg * (1 - win) + (patch + rng.normal(0, sigma, patch.shape)) * win
    m = np.zeros(img.shape, bool); m[y0:y0 + h, x0:x0 + w] = win > 0.25
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), m

def random_lung_point(lung, margin, rng):
    er = ndi.binary_erosion(lung, iterations=margin)
    idx = np.argwhere(er)
    if len(idx) == 0:
        idx = np.argwhere(lung)
    return idx[rng.integers(len(idx))]

def atk_copy_move(img, lung, rng, s=None):
    s = s or int(rng.integers(24, 40))
    a = random_lung_point(lung, s, rng)
    for _ in range(100):
        b = random_lung_point(lung, s, rng)
        if np.abs(a - b).max() > s + 4: break
    out = img.copy()
    out[b[0] - s // 2:b[0] - s // 2 + s, b[1] - s // 2:b[1] - s // 2 + s] = img[a[0] - s // 2:a[0] - s // 2 + s, a[1] - s // 2:a[1] - s // 2 + s]
    m = np.zeros(img.shape, bool); m[b[0] - s // 2:b[0] - s // 2 + s, b[1] - s // 2:b[1] - s // 2 + s] = True
    return out, m

def atk_inpaint(img, lung, nod, rng):
    cy, cx, r = nod
    h = int(2 * r + 16) | 1
    mk = np.zeros(img.shape, np.uint8); cv2.circle(mk, (cx, cy), int(r + 3), 1, -1)
    lo, hi = -1024.0, 400.0
    u8 = np.clip((img.astype(np.float32) - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
    inp = cv2.inpaint(u8, mk, 5, cv2.INPAINT_TELEA).astype(np.float32) / 255 * (hi - lo) + lo
    patch = ndi.gaussian_filter(inp, 1.0)[cy - h // 2:cy - h // 2 + h, cx - h // 2:cx - h // 2 + h]
    return blend_patch(img, patch, cy, cx, mad_sigma(img, lung), rng)

def atk_noise(img, rng):
    sig = rng.uniform(4, 12)
    body = img > -900
    out = img.astype(np.float32) + rng.normal(0, sig, img.shape) * body
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), body

In [ ]:
# --- FGSM : nécessite un classifieur de nodules (cible de l'attaque adversariale) ---
import timm

def hu_to_net(x):  # normalisation utilisée par tous les réseaux « forensiques » (pas d'écrêtage des poumons)
    return x / 1000.0

class NoduleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = timm.create_model("resnet18", pretrained=True, in_chans=1, num_classes=1)
    def forward(self, x):
        return self.net(x).squeeze(1)

def atk_fgsm(img, lung, model, eps_hu=8.0, P=64):
    # Perturbation FGSM (éq. 2.4) calculée patch par patch sur le poumon, poussant vers « nodule ».
    model.eval()
    out = img.astype(np.float32).copy()
    ys, xs = np.where(lung)
    if len(ys) == 0:
        return img, np.zeros(img.shape, bool)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    coords = [(y, x) for y in range(y0, max(y0 + 1, y1 - P), P) for x in range(x0, max(x0 + 1, x1 - P), P)]
    batch = np.stack([out[y:y + P, x:x + P] for y, x in coords])[:, None]
    t = torch.tensor(hu_to_net(batch).astype(np.float32), device=DEVICE, requires_grad=True)
    loss = F.binary_cross_entropy_with_logits(model(t), torch.ones(len(coords), device=DEVICE))
    loss.backward()
    g = -t.grad.sign().cpu().numpy()[:, 0]   # descente sur la perte « nodule »
    for (y, x), gi in zip(coords, g):
        out[y:y + P, x:x + P] += eps_hu * gi * lung[y:y + P, x:x + P]
    return np.clip(np.round(out), HU_MIN, HU_MAX).astype(np.int16), lung.copy()

### 3 bis. Cache des coupes candidates et banques de patchs (un seul passage sur les DICOM)

Lecture de chaque patient une seule fois : coupes candidates (HU int16), masques pulmonaires, nodules présents dans la coupe.
Les scanners sont téléchargés par lots de 8 en parallèle pendant que le lot précédent est lu, puis supprimés du disque.
Les patchs servant à entraîner les **outils de l'attaquant** (GAN, DDPM, classifieur cible de FGSM) proviennent **uniquement des patients d'entraînement**.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
P = CFG["PATCH"]
CACHE = {}          # (pid, z) -> dict(img, lung, nods)
NOD_PATCHES, LUNG_PATCHES = [], []
rng = np.random.default_rng(CFG["SEED"] + 1)
t0 = time.time(); FAILED = []
_dl = ThreadPoolExecutor(8)
_fut = {pid: _dl.submit(ensure_dicom, pid) for pid in patients[:8]}
for i, pid in enumerate(patients):
    for nxt in patients[i + 1:i + 9]:                      # pré-téléchargement des 8 suivants
        if nxt not in _fut: _fut[nxt] = _dl.submit(ensure_dicom, nxt)
    try:
        _fut.pop(pid).result()
        vol, nods, ps, st = load_patient(pid)
    except Exception as e:
        print("échec", pid, e); FAILED.append(pid); drop_dicom(pid); continue
    drop_dicom(pid)
    for z in candidate_slices(vol, nods, CFG["SLICES_PER_PATIENT"], rng):
        img = vol[:, :, z].copy()
        lung = lung_mask_slice(img)
        if lung.sum() < 4000:
            continue
        nz = [(r, c, rad) for (r, c, zz, rad) in nods if abs(zz - z) <= 1 and 40 < r < 472 and 40 < c < 472]
        CACHE[(pid, z)] = dict(img=img, lung=lung, nods=nz)
        if SPLIT[pid] == "train":
            for (r, c, rad) in nz:
                NOD_PATCHES.append(img[r - P // 2:r + P // 2, c - P // 2:c + P // 2])
            er_idx = np.argwhere(ndi.binary_erosion(lung, iterations=P // 2))
            for _ in range(6 if len(er_idx) else 0):
                y, x = er_idx[rng.integers(len(er_idx))]
                if all(abs(y - r) > P and abs(x - c) > P for (r, c, _) in nz):
                    pt = img[y - P // 2:y + P // 2, x - P // 2:x + P // 2]
                    if pt.shape == (P, P):
                        LUNG_PATCHES.append(pt)
    if i % 20 == 0:
        print(f"{i}/{len(patients)} patients, {len(CACHE)} coupes, {time.time() - t0:.0f}s", flush=True)
_dl.shutdown()
patients = [p for p in patients if p not in FAILED]
NOD_PATCHES = np.array([p for p in NOD_PATCHES if p.shape == (P, P)], np.int16)
LUNG_PATCHES = np.array(LUNG_PATCHES, np.int16)
# une coupe réelle avec nodule, conservée pour les figures du §12
try:
    np.save("real_slice.npy", CACHE[next(k for k, c in CACHE.items() if c["nods"])]["img"])
except StopIteration:
    pass
print("patients lus :", len(patients), "| échecs :", len(FAILED))
print("coupes en cache :", len(CACHE), "| patchs nodule :", NOD_PATCHES.shape, "| patchs poumon :", LUNG_PATCHES.shape)
chrono("cache des coupes construit")

### 3 ter. Classifieur de nodules (cible de l'attaque FGSM)

In [ ]:
def augment_d4(x):
    k = np.random.randint(4); x = np.rot90(x, k, axes=(-2, -1))
    return np.flip(x, -1) if np.random.rand() < 0.5 else x

Xn = np.concatenate([NOD_PATCHES, LUNG_PATCHES[:len(NOD_PATCHES) * 3]]).astype(np.float32)
yn = np.concatenate([np.ones(len(NOD_PATCHES)), np.zeros(min(len(LUNG_PATCHES), len(NOD_PATCHES) * 3))]).astype(np.float32)
nodcls = NoduleClassifier().to(DEVICE)
opt = torch.optim.Adam(nodcls.parameters(), 1e-4)
for ep in range(CFG["NODCLS_EPOCHS"]):
    perm = np.random.permutation(len(Xn)); tot = 0
    for b in range(0, len(perm), 64):
        idx = perm[b:b + 64]
        xb = torch.tensor(hu_to_net(np.stack([augment_d4(Xn[j]).copy() for j in idx]))[:, None], device=DEVICE)
        loss = F.binary_cross_entropy_with_logits(nodcls(xb), torch.tensor(yn[idx], device=DEVICE))
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    print(f"époque {ep}: perte {tot / len(perm):.4f}")

## 4. GAN d'inpainting conditionnel (pix2pix) — injection et suppression de nodules

Entrée du générateur : patch $64\times64$ dont le carré central $32\times32$ est masqué, le masque, et une carte de condition (1 = générer un nodule, 0 = générer du parenchyme sain). Perte : $\mathcal{L}_{cGAN} + 100\,\lVert y - G(x_m)\rVert_1$ (éq. 2.5).

In [ ]:
WIN_LO, WIN_HI = -1100.0, 300.0
def to_gan(x):   return (np.clip((x - WIN_LO) / (WIN_HI - WIN_LO), 0, 1) * 2 - 1).astype(np.float32)
def from_gan(y): return (y + 1) / 2 * (WIN_HI - WIN_LO) + WIN_LO

CENTER = np.zeros((P, P), np.float32); CENTER[P // 4:3 * P // 4, P // 4:3 * P // 4] = 1

def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2, True))

class UNetG(nn.Module):
    def __init__(self, cin=3, b=48):
        super().__init__()
        self.e1, self.e2, self.e3, self.bt = cbr(cin, b), cbr(b, 2 * b), cbr(2 * b, 4 * b), cbr(4 * b, 8 * b)
        self.u3, self.d3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, 2), cbr(8 * b, 4 * b)
        self.u2, self.d2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, 2), cbr(4 * b, 2 * b)
        self.u1, self.d1 = nn.ConvTranspose2d(2 * b, b, 2, 2), cbr(2 * b, b)
        self.out = nn.Conv2d(b, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2)); bt = self.bt(F.max_pool2d(e3, 2))
        d3 = self.d3(torch.cat([self.u3(bt), e3], 1)); d2 = self.d2(torch.cat([self.u2(d3), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return torch.tanh(self.out(d1))

class PatchD(nn.Module):
    def __init__(self, cin=4, b=64):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(cin, b, 4, 2, 1), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(b, 2 * b, 4, 2, 1), nn.BatchNorm2d(2 * b), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(2 * b, 4 * b, 4, 2, 1), nn.BatchNorm2d(4 * b), nn.LeakyReLU(0.2, True),
                                 nn.Conv2d(4 * b, 1, 3, 1, 1))
    def forward(self, x): return self.net(x)

def gan_inputs(y, cond):
    # y : (B,1,P,P) dans [-1,1] ; cond : (B,) 0/1
    m = torch.tensor(CENTER, device=y.device)[None, None].expand_as(y)
    c = cond.view(-1, 1, 1, 1).float().expand_as(y)
    return torch.cat([y * (1 - m), m, c], 1), m

G, D = UNetG().to(DEVICE), PatchD().to(DEVICE)
oG = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5, 0.999)); oD = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5, 0.999))
Xg = np.concatenate([NOD_PATCHES, LUNG_PATCHES]); cg = np.concatenate([np.ones(len(NOD_PATCHES)), np.zeros(len(LUNG_PATCHES))])
w = np.where(cg == 1, len(cg) / (2 * cg.sum()), len(cg) / (2 * (1 - cg).sum())); w /= w.sum()   # rééquilibrage
bce = nn.BCEWithLogitsLoss()
steps = max(1, len(Xg) // 64)
for ep in range(CFG["GAN_EPOCHS"]):
    for _ in range(steps):
        idx = np.random.choice(len(Xg), 64, p=w)
        y = torch.tensor(to_gan(np.stack([augment_d4(Xg[j]).copy() for j in idx]))[:, None], device=DEVICE)
        cond = torch.tensor(cg[idx], device=DEVICE)
        x, m = gan_inputs(y, cond)
        fake = G(x); comp = y * (1 - m) + fake * m
        # discriminateur
        dr, df = D(torch.cat([x, y], 1)), D(torch.cat([x, comp.detach()], 1))
        lD = 0.5 * (bce(dr, torch.ones_like(dr)) + bce(df, torch.zeros_like(df)))
        oD.zero_grad(); lD.backward(); oD.step()
        # générateur
        df = D(torch.cat([x, comp], 1))
        lG = bce(df, torch.ones_like(df)) + 100 * F.l1_loss(fake * m, y * m)
        oG.zero_grad(); lG.backward(); oG.step()
    print(f"GAN époque {ep}: L_D={lD.item():.3f} L_G={lG.item():.3f}")
torch.save(G.state_dict(), OUT / "gan_G.pt")

In [ ]:
@torch.no_grad()
def gan_fill(patch_hu, cond):
    G.eval()
    y = torch.tensor(to_gan(patch_hu)[None, None], device=DEVICE)
    x, m = gan_inputs(y, torch.tensor([cond], device=DEVICE))
    out = (y * (1 - m) + G(x) * m)[0, 0].cpu().numpy()
    return from_gan(out)

def paste_center(img, lung, new_patch, cy, cx, rng):
    # Recolle la zone centrale générée (40x40) avec fenêtre gaussienne + bruit de camouflage.
    c = new_patch[P // 2 - 20:P // 2 + 20, P // 2 - 20:P // 2 + 20]
    return blend_patch(img, c, cy, cx, mad_sigma(img, lung), rng)

def atk_gan_inject(img, lung, rng):
    cy, cx = random_lung_point(lung, P // 2 + 2, rng)
    return paste_center(img, lung, gan_fill(img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], 1), cy, cx, rng)

def atk_gan_remove(img, lung, nod, rng):
    cy, cx, _ = nod
    return paste_center(img, lung, gan_fill(img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], 0), cy, cx, rng)

# Contrôle visuel
k = min(4, len(NOD_PATCHES))
fig, ax = plt.subplots(2, k, figsize=(3 * k, 6))
for j in range(k):
    ax[0, j].imshow(to_gan(NOD_PATCHES[j]), cmap="gray"); ax[0, j].set_title("réel (nodule)")
    ax[1, j].imshow(to_gan(gan_fill(NOD_PATCHES[j], 0)), cmap="gray"); ax[1, j].set_title("GAN : suppression")
for a in ax.ravel(): a.axis("off")
plt.show()

## 5. Modèle de diffusion (DDPM) et inpainting de type RePaint

DDPM à $T=1000$ pas (planning linéaire de $\beta_t$), perte $\lVert\epsilon-\epsilon_\theta(x_t,t)\rVert^2$ (éq. 1.7), entraîné **uniquement sur des patchs pulmonaires authentiques**.
**Attaquant et défenseur disposent de deux modèles distincts** (`SEPARATE_DEFENDER_DDPM=True`), entraînés sur deux moitiés disjointes des patients d'entraînement : le défenseur ne connaît donc pas le générateur de l'attaquant, ce qui évite de surestimer la branche diffusion.

In [ ]:
SEPARATE_DEFENDER_DDPM = True
T = CFG["T_DIFF"]
betas = torch.linspace(1e-4, 0.02, T, device=DEVICE)
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

class TEmb(nn.Module):
    def __init__(self, d=128):
        super().__init__(); self.d = d
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.SiLU(), nn.Linear(4 * d, 4 * d))
    def forward(self, t):
        half = self.d // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        a = t.float()[:, None] * f[None]
        return self.mlp(torch.cat([a.sin(), a.cos()], 1))

class ResB(nn.Module):
    def __init__(self, i, o, td=512):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(8, i), nn.Conv2d(i, o, 3, padding=1)
        self.t = nn.Linear(td, o)
        self.n2, self.c2 = nn.GroupNorm(8, o), nn.Conv2d(o, o, 3, padding=1)
        self.sk = nn.Conv2d(i, o, 1) if i != o else nn.Identity()
    def forward(self, x, te):
        h = self.c1(F.silu(self.n1(x))) + self.t(te)[:, :, None, None]
        return self.c2(F.silu(self.n2(h))) + self.sk(x)

class DiffUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.te = TEmb(128); self.inc = nn.Conv2d(1, 64, 3, padding=1)
        self.d1, self.d2, self.d3, self.mid = ResB(64, 64), ResB(64, 128), ResB(128, 256), ResB(256, 256)
        self.u3, self.u2, self.u1 = ResB(512, 128), ResB(256, 64), ResB(128, 64)
        self.no, self.out = nn.GroupNorm(8, 64), nn.Conv2d(64, 1, 3, padding=1)
    def forward(self, x, t):
        te = self.te(t); x0 = self.inc(x)
        h1 = self.d1(x0, te); h2 = self.d2(F.avg_pool2d(h1, 2), te); h3 = self.d3(F.avg_pool2d(h2, 2), te)
        m = self.mid(F.avg_pool2d(h3, 2), te)
        u = self.u3(torch.cat([F.interpolate(m, scale_factor=2), h3], 1), te)
        u = self.u2(torch.cat([F.interpolate(u, scale_factor=2), h2], 1), te)
        u = self.u1(torch.cat([F.interpolate(u, scale_factor=2), h1], 1), te)
        return self.out(F.silu(self.no(u)))

def train_ddpm(patches, epochs, tag):
    net = DiffUNet().to(DEVICE); opt = torch.optim.AdamW(net.parameters(), 2e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    X = to_gan(patches)
    for ep in range(epochs):
        perm = np.random.permutation(len(X)); tot = 0
        for b in range(0, len(perm) - 63, 64):
            x0 = torch.tensor(np.stack([augment_d4(X[j]).copy() for j in perm[b:b + 64]])[:, None], device=DEVICE)
            t = torch.randint(0, T, (len(x0),), device=DEVICE); eps = torch.randn_like(x0)
            xt = abar[t].sqrt()[:, None, None, None] * x0 + (1 - abar[t]).sqrt()[:, None, None, None] * eps
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = F.mse_loss(net(xt, t), eps)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
        print(f"DDPM[{tag}] époque {ep}: perte {tot / max(1, len(perm) // 64):.4f}")
    torch.save(net.state_dict(), OUT / f"ddpm_{tag}.pt")
    return net.eval()

half = len(LUNG_PATCHES) // 2       # les patchs sont rangés patient par patient : les deux moitiés sont disjointes en patients (à un patient près)
ddpm_att = train_ddpm(LUNG_PATCHES[:half] if SEPARATE_DEFENDER_DDPM else LUNG_PATCHES, CFG["DDPM_EPOCHS"], "attaquant")
ddpm_def = train_ddpm(LUNG_PATCHES[half:], CFG["DDPM_EPOCHS"], "defenseur") if SEPARATE_DEFENDER_DDPM else ddpm_att
chrono("DDPM entraînés")

In [ ]:
@torch.no_grad()
def repaint(net, patch_hu, mask):
    # Inpainting RePaint (sans rééchantillonnage) : la région connue est réinjectée bruitée à chaque pas.
    x0 = torch.tensor(to_gan(patch_hu)[None, None], device=DEVICE)
    m = torch.tensor(mask[None, None].astype(np.float32), device=DEVICE)    # 1 = à générer
    x = torch.randn_like(x0)
    for t in reversed(range(T)):
        tt = torch.full((1,), t, device=DEVICE, dtype=torch.long)
        eps = net(x, tt)
        mean = (x - betas[t] / (1 - abar[t]).sqrt() * eps) / alphas[t].sqrt()
        x_unk = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
        ab_prev = abar[t - 1] if t > 0 else torch.tensor(1.0, device=DEVICE)
        x_kn = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * torch.randn_like(x0) if t > 0 else x0
        x = m * x_unk + (1 - m) * x_kn
    return from_gan(x[0, 0].clamp(-1, 1).cpu().numpy())

def atk_diffusion(img, lung, rng, nod=None):
    if nod is not None:
        cy, cx, _ = nod
    else:
        cy, cx = random_lung_point(lung, P // 2 + 2, rng)
    new = repaint(ddpm_att, img[cy - P // 2:cy + P // 2, cx - P // 2:cx + P // 2], CENTER)
    return paste_center(img, lung, new, cy, cx, rng)

## 5 bis. Construction du jeu falsifié

Chaque coupe candidate reçoit soit aucune attaque (authentique, probabilité `P_AUTH`), soit **une** attaque tirée parmi celles compatibles (les suppressions exigent un nodule dans la coupe). Le masque de vérité terrain est conservé.

In [ ]:
ATTACKS = ["authentique", "copier_deplacer", "inpainting", "bruit", "fgsm", "gan_injection", "gan_suppression", "diffusion"]
NEEDS_NODULE = {"inpainting", "gan_suppression"}
GLOBAL_ATTACKS = {"bruit", "fgsm"}

def apply_attack(name, img, lung, nods, rng):
    if name == "authentique":     return img, np.zeros(img.shape, bool)
    if name == "copier_deplacer": return atk_copy_move(img, lung, rng)
    if name == "inpainting":      return atk_inpaint(img, lung, nods[0], rng)
    if name == "bruit":           return atk_noise(img, rng)
    if name == "fgsm":            return atk_fgsm(img, lung, nodcls)
    if name == "gan_injection":   return atk_gan_inject(img, lung, rng)
    if name == "gan_suppression": return atk_gan_remove(img, lung, nods[0], rng)
    if name == "diffusion":       return atk_diffusion(img, lung, rng, nods[0] if nods and rng.random() < 0.5 else None)
    raise ValueError(name)

rng = np.random.default_rng(CFG["SEED"] + 2)
DATA = {}   # (pid, z) -> dict(img, mask, lung, attack, split)
rows = []; ATK_FAIL = {}
for (pid, z), c in CACHE.items():
    if rng.random() < CFG["P_AUTH"]:
        atk = "authentique"
    else:
        pool = [a for a in ATTACKS[1:] if (a not in NEEDS_NODULE) or c["nods"]]
        atk = pool[rng.integers(len(pool))]
    try:
        out, m = apply_attack(atk, c["img"], c["lung"], c["nods"], rng)
    except Exception as e:     # bord d'image, etc. -> coupe laissée authentique
        ATK_FAIL[atk] = ATK_FAIL.get(atk, 0) + 1
        atk, out, m = "authentique", c["img"], np.zeros(c["img"].shape, bool)
    DATA[(pid, z)] = dict(img=out, mask=m, lung=c["lung"], attack=atk, split=SPLIT[pid])
    rows.append(dict(pid=pid, z=z, split=SPLIT[pid], attack=atk, label=int(atk != "authentique"), area=float(m.mean())))
MANIFEST = pd.DataFrame(rows)
MANIFEST.to_csv(OUT / "manifest.csv", index=False)
print(pd.crosstab(MANIFEST.attack, MANIFEST.split))
print("attaques échouées (coupes laissées authentiques) :", ATK_FAIL)
CACHE.clear()      # libère la mémoire : DATA contient désormais toutes les coupes
chrono("jeu falsifié construit")

## 6. Branche forensique

Le module `features.py` (identique à celui de la preuve de concept) fournit les résidus SRM/KV, l'autocorrélation (éq. 2.6), l'énergie spectrale par anneaux (éq. 2.7), la normalisation robuste intra-coupe (éq. 2.8) et la détection de duplication par hachage.
On l'applique ici sur des **patchs $64\times64$ au pas de 32** couvrant les poumons.

In [ ]:
%%writefile features.py
"""Extraction de caractéristiques forensiques (résidus de bruit, spectre, cohérence locale, duplication)."""
import numpy as np
from scipy import ndimage as ndi
from scipy.stats import kurtosis, skew

B = 16
K_SRM3 = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], np.float32) / 4.0
K_KV = np.array([[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2],
                 [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]], np.float32) / 12.0


def residuals(img):
    f = img.astype(np.float32)
    return {
        "med": f - ndi.median_filter(f, 3),
        "srm3": ndi.convolve(f, K_SRM3),
        "kv": ndi.convolve(f, K_KV),
        "dx": np.diff(f, axis=1, append=f[:, -1:]),
    }


def ac1(r):
    a = r - r.mean()
    d = (a * a).mean() + 1e-6
    return float((a[:, 1:] * a[:, :-1]).mean() / d), float((a[1:] * a[:-1]).mean() / d)


def band_energy(r, nb=6):
    F = np.abs(np.fft.fftshift(np.fft.fft2(r - r.mean()))) ** 2
    h, w = r.shape
    yy, xx = np.mgrid[:h, :w]
    rad = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2) / (min(h, w) / 2)
    e = np.array([F[(rad >= i / nb) & (rad < (i + 1) / nb)].mean() for i in range(nb)])
    return e / (e.sum() + 1e-9)


def lung_blocks(lungs):
    er = ndi.binary_erosion(lungs, iterations=2)
    out = []
    for y in range(0, lungs.shape[0] - B + 1, B // 2):
        for x in range(0, lungs.shape[1] - B + 1, B // 2):
            if er[y:y + B, x:x + B].all():
                out.append((y, x))
    return out


def block_feats(img, R, blocks):
    """Descripteurs locaux par bloc : niveau de bruit, autocorrélation, ratio spectral haut/moyen."""
    rows = []
    for y, x in blocks:
        r = R["med"][y:y + B, x:x + B]
        k = R["kv"][y:y + B, x:x + B]
        sig = 1.4826 * np.median(np.abs(r - np.median(r)))
        a1x, a1y = ac1(k)
        e = band_energy(k, 4)
        rows.append([sig, a1x, a1y, e[3] / (e[1] + 1e-9), k.std(), kurtosis(k.ravel())])
    return np.array(rows, np.float32)


def duplicate_map(img, body, b=8):
    """Détection de blocs strictement identiques (copier-déplacer) par hachage."""
    from numpy.lib.stride_tricks import sliding_window_view
    dup = np.zeros(img.shape, bool)
    W = sliding_window_view(np.ascontiguousarray(img), (b, b))
    H, Wd = W.shape[:2]
    flat = np.ascontiguousarray(W.reshape(H * Wd, b * b))
    yy, xx = np.divmod(np.arange(H * Wd), Wd)
    ok = body[yy + b // 2, xx + b // 2] & (flat.std(1) >= 3)
    idx = np.where(ok)[0]
    if len(idx) == 0:
        return dup
    v = flat[idx].view(np.dtype((np.void, flat.dtype.itemsize * b * b))).ravel()
    _, inv, cnt = np.unique(v, return_inverse=True, return_counts=True)
    for g in np.where(cnt > 1)[0]:
        mem = idx[inv == g]
        ys, xs = yy[mem], xx[mem]
        if (np.abs(ys - ys[0]).max() > b) or (np.abs(xs - xs[0]).max() > b):
            for y, x in zip(ys, xs):
                dup[y:y + b, x:x + b] = True
    return dup


def robust_z(v):
    med = np.median(v, 0)
    mad = 1.4826 * np.median(np.abs(v - med), 0) + 1e-6
    return (v - med) / mad


def image_features(img, lungs):
    R = residuals(img)
    body = img > -900
    feats = []
    for k in ("med", "srm3", "kv", "dx"):
        v = R[k][body]
        feats += [v.std(), kurtosis(v), skew(v)]
    lv = R["kv"][lungs]
    feats += list(ac1(np.where(lungs, R["kv"], 0)))
    # spectre du résidu dans une fenêtre pulmonaire centrale
    ys, xs = np.where(ndi.binary_erosion(lungs, iterations=4))
    cy, cx = int(np.median(ys)), int(np.median(xs))
    win = R["kv"][max(cy - 32, 0):cy + 32, max(cx - 32, 0):cx + 32]
    feats += list(band_energy(win, 6))
    blocks = lung_blocks(lungs)
    bf = block_feats(img, R, blocks)
    z = np.abs(robust_z(bf))
    feats += list(z.max(0)) + list(np.percentile(z, 95, axis=0)) + list(bf.std(0) / (np.abs(bf.mean(0)) + 1e-6))
    dup = duplicate_map(img, body)
    feats += [dup.mean() * 1000, float(dup.any())]
    return np.array(feats, np.float32), (blocks, bf, dup)


def intensity_features(img, lungs):
    """Ligne de base « contenu » : statistiques d'intensité HU uniquement."""
    v = img[lungs].astype(np.float32)
    b = img[img > -900].astype(np.float32)
    return np.array([v.mean(), v.std(), *np.percentile(v, [5, 25, 50, 75, 95]),
                     b.mean(), b.std(), *np.histogram(b, bins=12, range=(-1000, 1000), density=True)[0]], np.float32)

In [ ]:
import importlib, features as FZ
importlib.reload(FZ)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, accuracy_score, precision_score, recall_score

S = CFG["STRIDE"]

def grid_coords(lung):
    out = []
    for y in range(0, lung.shape[0] - P + 1, S):
        for x in range(0, lung.shape[1] - P + 1, S):
            if lung[y:y + P, x:x + P].mean() > 0.5:
                out.append((y, x))
    return out

def patch_desc(R, img, y, x):
    r = R["med"][y:y + P, x:x + P]; k = R["kv"][y:y + P, x:x + P]
    sig = 1.4826 * np.median(np.abs(r - np.median(r)))
    a1x, a1y = FZ.ac1(k); e = FZ.band_energy(k, 6)
    blk = img[y:y + P, x:x + P].astype(np.float32)
    return [sig, a1x, a1y, e[5] / (e[2] + 1e-9), k.std(), kurtosis(k.ravel()), *e, blk.mean(), blk.std()]

def slice_patch_features(img, lung, coords=None):
    coords = coords if coords is not None else grid_coords(lung)
    if not coords:
        return coords, np.zeros((0, 27), np.float32)
    R = FZ.residuals(img)
    f = np.array([patch_desc(R, img, y, x) for y, x in coords], np.float32)
    z = np.abs(FZ.robust_z(f[:, :12]))
    dup = FZ.duplicate_map(img, img > -900)
    d = np.array([[dup[y:y + P, x:x + P].mean()] for y, x in coords], np.float32)
    return coords, np.nan_to_num(np.hstack([f, z, d]))

def patch_labels(mask, coords, attack):
    if attack in GLOBAL_ATTACKS:
        return np.ones(len(coords), int)
    return np.array([mask[y:y + P, x:x + P].mean() > 0.3 for y, x in coords], int)

# Extraction pour toutes les coupes (parallélisée sur les cœurs CPU), avec descripteurs globaux de la coupe
def _pf_job(key):
    d = DATA[key]
    coords, f = slice_patch_features(d["img"], d["lung"])
    return coords, f, FZ.image_features(d["img"], d["lung"])[0]

t0 = time.time()
_keys = list(DATA)
PF = {}
for key, (coords, f, g) in zip(_keys, pmap(_pf_job, _keys)):
    PF[key] = dict(coords=coords, f=f, g=g, y=patch_labels(DATA[key]["mask"], coords, DATA[key]["attack"]))
print(f"descripteurs de patchs : {time.time() - t0:.0f}s pour {len(PF)} coupes")

def stack_split(split):
    keys = [k for k in PF if DATA[k]["split"] == split and len(PF[k]["coords"])]
    X = np.vstack([PF[k]["f"] for k in keys]); y = np.concatenate([PF[k]["y"] for k in keys])
    owner = np.concatenate([[i] * len(PF[k]["coords"]) for i, k in enumerate(keys)])
    return keys, X, y, owner

trk, Xtr, ytr, _ = stack_split("train")
# sous-échantillonnage des négatifs (3:1)
neg = np.where(ytr == 0)[0]; pos = np.where(ytr == 1)[0]
sel = np.concatenate([pos, np.random.choice(neg, min(len(neg), 3 * len(pos)), replace=False)])
forensic_clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, class_weight="balanced", random_state=0)
forensic_clf.fit(Xtr[sel], ytr[sel])
vak, Xva, yva, _ = stack_split("val")
print("AUC patchs (val) — branche forensique :", round(roc_auc_score(yva, forensic_clf.predict_proba(Xva)[:, 1]), 4))
chrono("branche forensique entraînée")

## 7. Branche CNN : couche contrainte de Bayar + SRM → EfficientNet-B0 → décodeur U-Net

* Couche de Bayar (éq. 1.9) : poids centraux fixés à −1, somme des voisins = 1, reprojection à chaque passe avant.
* 3 résidus SRM fixes (KV $5\times5$, SRM $3\times3$, différence horizontale) + l'image (HU/1000) → 7 canaux.
* Perte : BCE (patch) + Dice (carte), éq. 2.9. Augmentation : D4, flou, ré-échantillonnage, bruit, export 8 bits.

In [ ]:
class BayarConv(nn.Module):
    def __init__(self, cin=1, cout=3, k=5):
        super().__init__(); self.k = k
        self.w = nn.Parameter(torch.randn(cout, cin, k, k) * 0.01)
        m = torch.ones(cout, cin, k, k); m[:, :, k // 2, k // 2] = 0
        c = torch.zeros(cout, cin, k, k); c[:, :, k // 2, k // 2] = -1
        self.register_buffer("mask", m); self.register_buffer("center", c)
    def forward(self, x):
        w = self.w * self.mask
        w = w / (w.sum(dim=(2, 3), keepdim=True) + 1e-8) + self.center
        return F.conv2d(x, w, padding=self.k // 2)

def srm_bank():
    kv = torch.tensor(FZ.K_KV); s3 = torch.zeros(5, 5); s3[1:4, 1:4] = torch.tensor(FZ.K_SRM3)
    d1 = torch.zeros(5, 5); d1[2, 2], d1[2, 3] = -1, 1
    return torch.stack([kv, s3, d1])[:, None]

class Up(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(cin + cskip, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(True),
                                  nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))

class ForensicUNet(nn.Module):
    def __init__(self, encoder="efficientnet_b0", pretrained=True):
        super().__init__()
        self.bayar = BayarConv(1, 3, 5)
        self.register_buffer("srm", srm_bank())
        self.enc = timm.create_model(encoder, features_only=True, pretrained=pretrained, in_chans=7)
        ch = self.enc.feature_info.channels()
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(ch[-1], 1))
        self.ups = nn.ModuleList()
        cur = ch[-1]
        for c in reversed(ch[:-1]):
            self.ups.append(Up(cur, c, max(32, c))); cur = max(32, c)
        self.seg = nn.Conv2d(cur, 1, 1)
    def forward(self, x):
        x7 = torch.cat([x, self.bayar(x), F.conv2d(x, self.srm, padding=2)], 1)
        feats = self.enc(x7)
        logit = self.cls(feats[-1]).squeeze(1)
        d = feats[-1]
        for up, sk in zip(self.ups, reversed(feats[:-1])):
            d = up(d, sk)
        seg = F.interpolate(self.seg(d), size=x.shape[-2:], mode="bilinear", align_corners=False)
        return logit, seg[:, 0]

def postprocess(img, kind):
    f = img.astype(np.float32)
    if kind == "blur":    f = ndi.gaussian_filter(f, 0.5)
    elif kind == "resample":
        h, w = f.shape; f = cv2.resize(cv2.resize(f, (int(w * 0.75), int(h * 0.75)), interpolation=cv2.INTER_LINEAR), (w, h), interpolation=cv2.INTER_LINEAR)
    elif kind == "noise": f = f + np.random.normal(0, 2, f.shape)
    elif kind == "8bit":
        lo, hi = -1350, 150; f = np.round(np.clip((f - lo) / (hi - lo), 0, 1) * 255) / 255 * (hi - lo) + lo
    return np.clip(np.round(f), HU_MIN, HU_MAX).astype(np.int16)

class PatchSet(Dataset):
    def __init__(self, split, neg_ratio=2, augment=False):
        self.items = []
        for k, pf in PF.items():
            if DATA[k]["split"] != split: continue
            for (y, x), lab in zip(pf["coords"], pf["y"]):
                self.items.append((k, y, x, int(lab)))
        self.augment = augment; self.neg_ratio = neg_ratio; self.resample()
    def resample(self):
        pos = [i for i in self.items if i[3] == 1]; neg = [i for i in self.items if i[3] == 0]
        random.shuffle(neg); self.cur = pos + neg[:self.neg_ratio * max(1, len(pos))]; random.shuffle(self.cur)
    def __len__(self): return len(self.cur)
    def __getitem__(self, i):
        k, y, x, lab = self.cur[i]; d = DATA[k]
        y0, x0 = max(0, y - 8), max(0, x - 8)          # marge pour que flou / ré-échantillonnage n'aient pas d'effet de bord
        big = d["img"][y0:y + P + 8, x0:x + P + 8]; mbig = d["mask"][y0:y + P + 8, x0:x + P + 8]
        if self.augment and random.random() < 0.4:
            big = postprocess(big, random.choice(["blur", "resample", "noise", "8bit"]))
        oy, ox = y - y0, x - x0
        pt = big[oy:oy + P, ox:ox + P].astype(np.float32); m = mbig[oy:oy + P, ox:ox + P].astype(np.float32)
        if d["attack"] in GLOBAL_ATTACKS: m = np.ones_like(m) * lab
        if self.augment:
            kk = random.randint(0, 3); pt, m = np.rot90(pt, kk), np.rot90(m, kk)
            if random.random() < 0.5: pt, m = pt[:, ::-1], m[:, ::-1]
        return torch.tensor(hu_to_net(pt.copy()))[None], torch.tensor(float(lab)), torch.tensor(m.copy())

def dice_loss(seg_logit, m, eps=1.0):
    p = torch.sigmoid(seg_logit)
    num = 2 * (p * m).sum((1, 2)) + eps; den = p.sum((1, 2)) + m.sum((1, 2)) + eps
    return (1 - num / den).mean()

cnn = ForensicUNet().to(DEVICE)
opt = torch.optim.Adam(cnn.parameters(), CFG["LR"])
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
tr_set, va_set = PatchSet("train", augment=True), PatchSet("val", neg_ratio=3)
best, patience = -1, 0
for ep in range(CFG["CNN_EPOCHS"]):
    cnn.train(); tr_set.resample()
    for xb, yb, mb in DataLoader(tr_set, CFG["BATCH"], shuffle=True, num_workers=NUM_WORKERS, drop_last=True):
        xb, yb, mb = xb.to(DEVICE), yb.to(DEVICE), mb.to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            logit, seg = cnn(xb)
            loss = F.binary_cross_entropy_with_logits(logit, yb) + dice_loss(seg.float(), mb)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    cnn.eval(); ps_, ys_ = [], []
    with torch.no_grad():
        for xb, yb, _ in DataLoader(va_set, 256, num_workers=NUM_WORKERS):
            ps_.append(torch.sigmoid(cnn(xb.to(DEVICE))[0]).cpu().numpy()); ys_.append(yb.numpy())
    auc = roc_auc_score(np.concatenate(ys_), np.concatenate(ps_))
    print(f"CNN époque {ep}: perte {loss.item():.4f} | AUC patchs val {auc:.4f}")
    if auc > best:
        best, patience = auc, 0; torch.save(cnn.state_dict(), OUT / "cnn_best.pt")
    else:
        patience += 1
        if patience >= 6: print("arrêt précoce"); break
cnn.load_state_dict(torch.load(OUT / "cnn_best.pt")); cnn.eval()
chrono(f"branche CNN entraînée (meilleure AUC patchs val {best:.4f})")

## 8. Branche diffusion : score d'erreur de reconstruction (éq. 2.10)

Pour chaque patch : bruitage jusqu'à $t^\ast$, prédiction de $\hat x_0$ en un pas, erreur quadratique moyenne. Branche **non supervisée** (aucun exemple falsifié utilisé).
L'erreur est normalisée au sein de la coupe (score z robuste) ; une région falsifiée peut se reconstruire **moins bien** (contenu étranger) ou **trop bien** (contenu généré, cf. DIRE) : le score de coupe utilise donc l'écart absolu $|z|$.

In [ ]:
@torch.no_grad()
def diffusion_score(patches_hu, net=None, t_star=None, n_draws=3, bs=256):
    net = net or ddpm_def; t_star = t_star or CFG["T_STAR"]
    out = []
    for b in range(0, len(patches_hu), bs):
        x0 = torch.tensor(to_gan(patches_hu[b:b + bs])[:, None], device=DEVICE)
        t = torch.full((len(x0),), t_star, device=DEVICE, dtype=torch.long)
        acc = torch.zeros(len(x0), device=DEVICE)
        for _ in range(n_draws):
            eps = torch.randn_like(x0)
            xt = abar[t_star].sqrt() * x0 + (1 - abar[t_star]).sqrt() * eps
            x0h = (xt - (1 - abar[t_star]).sqrt() * net(xt, t)) / abar[t_star].sqrt()
            acc += ((x0h - x0) ** 2).mean((1, 2, 3))
        out.append((acc / n_draws).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)

## 9. Fusion local → global et évaluation

* **Local** : pour chaque patch, $[p_{for}, p_{cnn}, z(s_{diff})]$ → régression logistique (apprise sur la validation).
* **Global** : agrégation (max, moyenne top-3, top-10, nombre > 0,5) + descripteurs globaux de la coupe → HGB (appris sur la validation), éq. 2.11.
* Test interne (patients jamais vus), **test externe CT-GAN** (générateur jamais vu) et robustesse aux post-traitements.

In [ ]:
@torch.no_grad()
def cnn_patch(img, coords):
    if not coords: return np.zeros(0), np.zeros((0, P, P))
    X = np.stack([img[y:y + P, x:x + P] for y, x in coords]).astype(np.float32)
    pr, sg = [], []
    for b in range(0, len(X), 256):
        lo, se = cnn(torch.tensor(hu_to_net(X[b:b + 256]))[:, None].to(DEVICE))
        pr.append(torch.sigmoid(lo).cpu().numpy()); sg.append(torch.sigmoid(se).cpu().numpy())
    return np.concatenate(pr), np.concatenate(sg)

def cpu_part(img, lung):
    coords, f = slice_patch_features(img, lung)
    return coords, f, FZ.image_features(img, lung)[0]

def gpu_part(img, coords, f, gfeat, keep_seg=True):
    if not coords:
        return None
    p_for = forensic_clf.predict_proba(f)[:, 1]
    p_cnn, seg = cnn_patch(img, coords)
    X = np.stack([img[y:y + P, x:x + P] for y, x in coords])
    s = diffusion_score(X)
    s_z = (s - np.median(s)) / (1.4826 * np.median(np.abs(s - np.median(s))) + 1e-9)
    return dict(coords=coords, p_for=p_for, p_cnn=p_cnn, seg=seg if keep_seg else None, s_diff=s_z, gfeat=gfeat)

def branch_outputs(img, lung):
    return gpu_part(img, *cpu_part(img, lung))

_JOB = {}
def _cpu_job(key):
    img, lung = _JOB[key]
    return cpu_part(img, lung)

def branch_outputs_many(images, keep_seg=True):
    # images : dict clé -> (img, lung). Partie CPU en parallèle, partie GPU séquentielle.
    global _JOB
    _JOB = images; keys = list(images)
    cpu = pmap(_cpu_job, keys)
    out = {k: gpu_part(images[k][0], *c, keep_seg=keep_seg) for k, c in zip(keys, cpu)}
    _JOB = {}
    return {k: v for k, v in out.items() if v is not None}

def agg(p):
    p = np.sort(p)[::-1]
    return [p[0], p[:3].mean(), p[:10].mean(), (p > 0.5).sum()] if len(p) else [0, 0, 0, 0]

def pixel_map(coords, vals, shape=(512, 512)):
    acc = np.zeros(shape); cnt = np.zeros(shape)
    for (y, x), v in zip(coords, vals):
        acc[y:y + P, x:x + P] += v; cnt[y:y + P, x:x + P] += 1
    return np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)

# descripteurs CPU déjà calculés au §6 : seule la partie GPU est exécutée ici
BO = {}
for k, d in DATA.items():
    if d["split"] in ("val", "test"):
        bo = gpu_part(d["img"], PF[k]["coords"], PF[k]["f"], PF[k]["g"])
        if bo is not None: BO[k] = bo
chrono("sorties des trois branches calculées (validation + test)")

# --- apprentissage de la fusion sur la VALIDATION uniquement ---
vkeys = [k for k in BO if DATA[k]["split"] == "val"]
L_X = np.vstack([np.c_[BO[k]["p_for"], BO[k]["p_cnn"], BO[k]["s_diff"], np.abs(BO[k]["s_diff"])] for k in vkeys])
L_y = np.concatenate([PF[k]["y"] for k in vkeys])
local_fuser = LogisticRegression(max_iter=2000, class_weight="balanced").fit(L_X, L_y)

def fused_local(bo):
    return local_fuser.predict_proba(np.c_[bo["p_for"], bo["p_cnn"], bo["s_diff"], np.abs(bo["s_diff"])])[:, 1]

def meta_features(bo):
    return np.r_[bo["gfeat"], agg(fused_local(bo)), agg(bo["p_for"]), agg(bo["p_cnn"]), agg(bo["s_diff"]), agg(np.abs(bo["s_diff"]))]

meta = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)
meta.fit(np.nan_to_num(np.array([meta_features(BO[k]) for k in vkeys])), np.array([int(DATA[k]["attack"] != "authentique") for k in vkeys]))

def slice_scores(bo):
    return dict(forensique=np.mean(np.sort(bo["p_for"])[::-1][:3]), cnn=np.mean(np.sort(bo["p_cnn"])[::-1][:3]),
                diffusion=np.mean(np.sort(np.abs(bo["s_diff"]))[::-1][:3]),
                hybride=meta.predict_proba(np.nan_to_num(meta_features(bo))[None])[0, 1])

def tpr_at(y, s, fpr_target=0.01):
    fpr, tpr, _ = roc_curve(y, s); return float(np.interp(fpr_target, fpr, tpr))

def dice(pred, gt):
    return 2 * (pred & gt).sum() / (pred.sum() + gt.sum() + 1e-9)

In [ ]:
# --- Test interne ---
tkeys = [k for k in BO if DATA[k]["split"] == "test"]
yt = np.array([int(DATA[k]["attack"] != "authentique") for k in tkeys])
SC = {b: np.array([slice_scores(BO[k])[b] for k in tkeys]) for b in ("forensique", "cnn", "diffusion", "hybride")}
RES = {}
for b, s in SC.items():
    RES[b] = dict(auc=roc_auc_score(yt, s), tpr1=tpr_at(yt, s))
# seuil hybride = 0,5 ; F1 / exactitude
pred_h = (SC["hybride"] >= 0.5).astype(int)
RES["hybride"].update(acc=accuracy_score(yt, pred_h), prec=precision_score(yt, pred_h), rec=recall_score(yt, pred_h), f1=f1_score(yt, pred_h))
# Dice par branche (attaques locales)
for b, getter in (("forensique", lambda bo: bo["p_for"]), ("cnn", None), ("hybride", lambda bo: fused_local(bo))):
    ds = []
    for k in tkeys:
        d = DATA[k]
        if d["attack"] in GLOBAL_ATTACKS or d["attack"] == "authentique": continue
        bo = BO[k]
        if b == "cnn":
            acc = np.zeros((512, 512)); cnt = np.zeros((512, 512))
            for (y, x), sg in zip(bo["coords"], bo["seg"]):
                acc[y:y + P, x:x + P] += sg; cnt[y:y + P, x:x + P] += 1
            pm = np.where(cnt > 0, acc / np.maximum(cnt, 1), 0)
        else:
            pm = pixel_map(bo["coords"], getter(bo))
        ds.append(dice(pm >= 0.5, d["mask"] & d["lung"]))
    RES[b]["dice"] = float(np.mean(ds)) if ds else float("nan")
# détection par type d'attaque (hybride)
per_attack = pd.Series({a: float(((SC["hybride"] >= 0.5) == (a != "authentique"))[np.array([DATA[k]["attack"] == a for k in tkeys])].mean())
                        for a in ATTACKS if any(DATA[k]["attack"] == a for k in tkeys)})
# AUC par type d'attaque (authentiques du test + coupes de ce type), pour chaque branche
AUC_ATK = {}
auth_idx = np.array([DATA[k]["attack"] == "authentique" for k in tkeys])
for a in ATTACKS[1:]:
    sel = np.array([DATA[k]["attack"] == a for k in tkeys])
    if sel.sum() >= 3 and auth_idx.sum() >= 3:
        m_ = sel | auth_idx
        AUC_ATK[a] = {b: float(roc_auc_score(yt[m_], SC[b][m_])) for b in SC}
print(pd.DataFrame(RES).T.round(3)); print(per_attack.round(3)); print(pd.DataFrame(AUC_ATK).T.round(3))
chrono("test interne terminé")

In [ ]:
# --- Test externe : jeu CT-GAN de l'UCI (générateur jamais vu) ---
# Le jeu fournit des scanners DICOM et un CSV de vérité terrain (type, uuid, slice, x, y).
# Types : FM = cancer injecté (faux malin), FB = cancer retiré (faux bénin), TM/TB = authentiques.
# Vérifiez les noms de colonnes et la convention d'indice de coupe en affichant quelques cas.
import pydicom
CT_ROOTS = [Path(CFG["CTGAN_ROOT"])] + [Path(p).parent for p in glob.glob("/kaggle/input/**/labels_exp1.csv", recursive=True)]
def _find_labels():
    out = []
    for root in CT_ROOTS:
        out += glob.glob(str(root / "**" / "labels_exp*.csv"), recursive=True)
    return sorted(set(out))

if not _find_labels() and CFG["DOWNLOAD_CTGAN"]:
    # Archive officielle (UCI id=520, Google Drive, ~6 Go) : téléchargée, extraite puis supprimée.
    try:
        import gdown, zipfile
        root = Path(CFG["CTGAN_ROOT"]); root.mkdir(parents=True, exist_ok=True)
        zp = root / "data.zip"
        gdown.download(id="1R0WD_5IZ3NlyCiOPf1Ex74nBnZYQegwr", output=str(zp), quiet=True)
        with zipfile.ZipFile(zp) as z: z.extractall(root)
        zp.unlink()
    except Exception as e:
        print("téléchargement CT-GAN impossible :", e)

def load_ctgan_scan(folder):
    # Même convention que le code de CT-GAN : coupes triées par ImagePositionPatient[2] croissant, valeurs en HU.
    sl = []
    for f in glob.glob(str(Path(folder) / "*")):
        try:
            sl.append(pydicom.dcmread(f))
        except Exception:
            pass
    sl = [s for s in sl if hasattr(s, "ImagePositionPatient") and hasattr(s, "PixelData")]
    sl.sort(key=lambda s: float(s.ImagePositionPatient[2]))
    return np.stack([dicom_to_hu_raw(s) for s in sl])

def dicom_to_hu_raw(ds):
    return (ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))).astype(np.int16)

EXT = None
labs = _find_labels()
if labs:
    ys_, ss_, ex_imgs, ex_y = [], {b: [] for b in SC}, {}, {}
    for lf in labs:
        lab = pd.read_csv(lf); lab.columns = [c.strip().lower() for c in lab.columns]
        base = Path(lf).parent
        for uuid, grp in lab.groupby("uuid"):
            folder = next(iter(glob.glob(str(base / "**" / str(uuid)), recursive=True)), None)
            if folder is None: continue
            try:
                arr = load_ctgan_scan(folder)
            except Exception as e:
                print("scan illisible", uuid, e); continue
            for j, r in grp.reset_index().iterrows():
                z = int(r["slice"])
                if not 0 <= z < len(arr): continue
                img = arr[z]
                ex_imgs[(lf, uuid, j)] = (img, lung_mask_slice(img))
                ex_y[(lf, uuid, j)] = int(str(r["type"]).strip().upper() in ("FM", "FB"))
    EXB = branch_outputs_many(ex_imgs, keep_seg=False)
    for k, bo in EXB.items():
        sc = slice_scores(bo)
        for b in SC: ss_[b].append(sc[b])
        ys_.append(ex_y[k])
    ys_ = np.array(ys_)
    if len(set(ys_)) == 2:
        EXT = {b: dict(auc=roc_auc_score(ys_, np.array(v)), tpr1=tpr_at(ys_, np.array(v)), n=len(ys_)) for b, v in ss_.items()}
        print(pd.DataFrame(EXT).T.round(3))
else:
    print("Jeu CT-GAN indisponible : test externe non réalisé.")
chrono("test externe CT-GAN terminé")

In [ ]:
# --- Robustesse aux post-traitements (hybride, test interne) ---
ROB = {"aucun": RES["hybride"]["auc"]}
for kind in ("blur", "resample", "noise", "8bit"):
    np.random.seed(0)
    imgs = {k: (postprocess(DATA[k]["img"], kind), DATA[k]["lung"]) for k in tkeys}
    bos = branch_outputs_many(imgs, keep_seg=False)
    s = [slice_scores(bos[k])["hybride"] if k in bos else 0.0 for k in tkeys]
    ROB[kind] = roc_auc_score(yt, s)
print(pd.Series(ROB).round(3))
chrono("robustesse évaluée")

In [ ]:
# --- Génération automatique du tableau 3.6 du mémoire ---
names = {"forensique": "Forensique (blocs + agrégation)", "cnn": "CNN contraint + EfficientNet-B0 / U-Net",
         "diffusion": "Diffusion (erreur de reconstruction)", "hybride": "Hybride (fusion des trois branches)"}
fmt = lambda v: "—" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f}".replace(".", ",")
T36 = pd.DataFrame([{
    "Branche": names[b],
    "AUC test interne": fmt(RES[b]["auc"]),
    "AUC CT-GAN (externe)": fmt(EXT[b]["auc"]) if EXT else "non disponible",
    "TPR à FPR = 1 %": fmt(RES[b]["tpr1"]),
    "Dice moyen": fmt(RES[b].get("dice")) if b != "diffusion" else "—"} for b in names])
print(T36.to_markdown(index=False))
T36.to_csv(OUT / "tableau_3_6.csv", index=False)
INFO = dict(patients=len(patients), coupes=len(DATA), coupes_test=len(tkeys), falsifiees_test=int(yt.sum()),
            separate_defender_ddpm=SEPARATE_DEFENDER_DDPM, gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU",
            duree_min=round((time.time() - T_START) / 60, 1))
json.dump(dict(INFO=INFO, RES=RES, EXT=EXT, ROB=ROB, per_attack=per_attack.to_dict(), auc_par_attaque=AUC_ATK,
               attaques_echouees=ATK_FAIL), open(OUT / "resultats_lidc.json", "w"), indent=1, default=float)
print(INFO)

### Visualisation : cartes de localisation sur le test

In [ ]:
ex = [k for k in tkeys if DATA[k]["attack"] not in GLOBAL_ATTACKS | {"authentique"}][:4]
fig, ax = plt.subplots(len(ex), 3, figsize=(10, 3.4 * len(ex)))
for i, k in enumerate(ex):
    d, bo = DATA[k], BO[k]; w = np.clip((d["img"] + 1350) / 1500, 0, 1)
    ax[i, 0].imshow(w, cmap="gray"); ax[i, 0].set_title(f"{d['attack']} — score {slice_scores(bo)['hybride']:.2f}")
    ax[i, 1].imshow(w, cmap="gray"); ax[i, 1].contour(d["mask"], [0.5], colors="orange"); ax[i, 1].set_title("vérité terrain")
    ax[i, 2].imshow(w, cmap="gray"); ax[i, 2].imshow(np.ma.masked_less(pixel_map(bo["coords"], fused_local(bo)), 0.05), cmap="magma", vmin=0, vmax=1, alpha=0.7)
    ax[i, 2].set_title("carte fusionnée")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "localisation_lidc.png", dpi=150); plt.show()

## 10. Module de sécurité pour les hôpitaux

1. Empreinte **SHA-256** des pixels + attributs critiques, comparée à l'empreinte enregistrée à l'acquisition.
2. Détecteur hybride → score, type d'attaque probable, carte.
3. Politique : **conforme** (< τ₁, choisi pour 1 % de faux positifs sur la validation), **à vérifier** (τ₁–τ₂), **alerte** (> τ₂ ou empreinte discordante).
4. **Journal d'audit chaîné** (chaque entrée contient l'empreinte de la précédente).
5. Notification radiologue / RSSI.

In [ ]:
import pydicom, datetime, getpass
CRITICAL_TAGS = ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "RescaleSlope", "RescaleIntercept", "Rows", "Columns"]

def dicom_fingerprint(ds):
    h = hashlib.sha256(ds.PixelData)
    for t in CRITICAL_TAGS:
        h.update(str(getattr(ds, t, "")).encode())
    return h.hexdigest()

def dicom_to_hu(ds):
    return (ds.pixel_array.astype(np.float32) * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))).astype(np.int16)

class AuditLog:
    def __init__(self, path):
        self.path = Path(path); self.path.touch(exist_ok=True)
    def _last(self):
        lines = self.path.read_text().strip().splitlines()
        return json.loads(lines[-1])["entry_hash"] if lines else "0" * 64
    def append(self, record):
        rec = dict(record, prev_hash=self._last(), time=datetime.datetime.now().isoformat(timespec="seconds"))
        rec["entry_hash"] = hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest()
        with self.path.open("a") as f: f.write(json.dumps(rec) + "\n")
        return rec
    def verify(self):
        prev = "0" * 64
        for i, line in enumerate(self.path.read_text().strip().splitlines()):
            rec = json.loads(line); h = rec.pop("entry_hash")
            if rec["prev_hash"] != prev or hashlib.sha256(json.dumps(rec, sort_keys=True).encode()).hexdigest() != h:
                return False, i
            prev = h
        return True, None

# Seuils calibrés sur la validation : τ1 -> 1 % de FP, τ2 -> 0,1 % de FP (ou 0,9 si trop peu de données)
val_auth = np.array([slice_scores(BO[k])["hybride"] for k in vkeys if DATA[k]["attack"] == "authentique"])
TAU1 = float(np.quantile(val_auth, 0.99)); TAU2 = float(max(np.quantile(val_auth, 0.999), min(0.9, TAU1 + 0.2)))
AUDIT = AuditLog(OUT / "audit_log.jsonl")
REGISTRY = {}   # empreintes enregistrées à l'acquisition (SOPInstanceUID -> sha256), à brancher sur la base de l'établissement

def verify_dicom(path, user="service-verif"):
    ds = pydicom.dcmread(path)
    fp = dicom_fingerprint(ds); ref = REGISTRY.get(str(getattr(ds, "SOPInstanceUID", "")))
    img = dicom_to_hu(ds); lung = lung_mask_slice(img)
    bo = branch_outputs(img, lung)
    score = slice_scores(bo)["hybride"] if bo else 0.0
    pmap = pixel_map(bo["coords"], fused_local(bo), img.shape) if bo else np.zeros(img.shape)
    hash_ok = (ref is None) or (ref == fp)
    verdict = "ALERTE" if (not hash_ok or score > TAU2) else ("A VERIFIER" if score > TAU1 else "CONFORME")
    rec = AUDIT.append(dict(sop=str(getattr(ds, "SOPInstanceUID", "")), sha256=fp, hash_ok=hash_ok, score=round(float(score), 4),
                            verdict=verdict, user=user, tau1=round(TAU1, 4), tau2=round(TAU2, 4)))
    return verdict, score, img, pmap, rec

print("τ1 =", round(TAU1, 3), "| τ2 =", round(TAU2, 3)); print("Journal intègre :", AUDIT.verify())

In [ ]:
# Interface de démonstration (Gradio) — à exécuter sur un poste du réseau interne uniquement
try:
    import gradio as gr
except ImportError:
    gr = None
    print("Gradio non installé : interface de démonstration ignorée (pip install gradio).")
def ui(file):
    verdict, score, img, pmap, rec = verify_dicom(file.name)
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    w = np.clip((img + 1350) / 1500, 0, 1)
    ax[0].imshow(w, cmap="gray"); ax[0].set_title("Coupe"); ax[1].imshow(w, cmap="gray")
    ax[1].imshow(np.ma.masked_less(pmap, 0.05), cmap="magma", vmin=0, vmax=1, alpha=0.7); ax[1].set_title("Carte de falsification")
    for a in ax: a.axis("off")
    return f"{verdict} (score {score:.3f})", fig, json.dumps(rec, indent=1)
if gr is not None:
    demo = gr.Interface(fn=ui, inputs=gr.File(label="Fichier DICOM"), outputs=[gr.Textbox(label="Verdict"), gr.Plot(), gr.Code(label="Entrée du journal d'audit")],
                        title="Vérification d'authenticité des images CT")
# demo.launch(share=False)

**Intégration PACS (Orthanc).** Orthanc déclenche un script Python à chaque instance reçue (`OnStoredInstance`) ; le script récupère le fichier via `GET /instances/{id}/file`, appelle `verify_dicom`, puis écrit le verdict dans les métadonnées de l'instance (`PUT /instances/{id}/metadata/...`) et notifie en cas d'alerte. Les analyses restent **sur site** : aucune image ne quitte l'établissement (loi n° 2024/017).

```python
# greffon Python d'Orthanc (extrait)
import orthanc, io
def OnChange(changeType, level, resource):
    if changeType == orthanc.ChangeType.NEW_INSTANCE:
        dcm = orthanc.RestApiGet(f"/instances/{resource}/file")
        verdict, score, *_ = verify_dicom(io.BytesIO(dcm))
        orthanc.RestApiPut(f"/instances/{resource}/metadata/1024", verdict.encode())
orthanc.RegisterOnChangeCallback(OnChange)
```